# `nisar_access_subset` — DPS job runner (bounding-box driven)

Discover NISAR **GCOV** granules over an area of interest with `earthaccess` (CMR), and
submit one MAAP DPS job per granule to the `nisar_access_subset` OGC process. The worker
downloads each granule itself using MAAP/ASF temporary credentials (S3-preferred), reads the
requested variables from `/science/LSAR/GCOV/grids/frequencyA`, optionally subsets by a
bounding box, and writes an intermediate **Zarr** store (plus a small `manifest.json`). No
pre-staging is needed.

The algorithm's `ram_min` is 8 GB, so defaults here target the **`maap-dps-worker-8gb`**
queue. For a quick plumbing check, use **`maap-dps-sandbox`** (8 GB, 10-min cap); a heavy
granule may need **`maap-dps-worker-16gb`**.

Run this in a **MAAP Hub (OGC) workspace** so maap-py v5 is available.

**Prereqs**
- The `nisar_access_subset` OGC process is registered (see `algorithm.yml` / the repo `README.md`).
- `earthaccess` installed (`pip install earthaccess`) and an Earthdata Login.

In [ ]:
# One-time in a fresh workspace:
# %pip install earthaccess
import earthaccess
import pandas as pd
import datetime, json, os, time
from maap.maap import MAAP

maap = MAAP()

# NISAR provisional/beta collections are auth-gated: you MUST log in to Earthdata or searches
# return 0 results. Run once per workspace.
auth = earthaccess.login(strategy="interactive")

## 1. Area of interest

The bounding box `(W, S, E, N)` used for CMR discovery is set in the search cell below (§2).
The per-job spatial subset is a separate bbox passed to the worker in §5 (`bbox` + `bbox_crs`),
which the worker reprojects into the granule's native CRS before slicing.

## 2. Search for NISAR GCOV granules (earthaccess / CMR)

NISAR data is discovered via **`earthaccess`** (CMR), not asf-search, and requires an
Earthdata login (done above).

- `short_name="NISAR_L2_GCOV_BETA_V1"` — the GCOV collection the algorithm defaults to
- `bounding_box=(W, S, E, N)` — a numeric tuple (not a WKT string)
- `temporal=(start, end)`

Each result exposes `data_links()` (HTTPS) and `data_links(access="direct")` (`s3://`).

In [ ]:
# CMR discovery bounding box (W, S, E, N). Adjust to your AOI.
BBOX = (-102.4704, -75.5511, -99.4242, -74.7848)

# Acquisition window (adjust as needed).
START = "2025-10-01T00:00:00Z"
END   = "2025-10-31T23:59:59Z"

SHORT_NAME = "NISAR_L2_GCOV_BETA_V1"

results = earthaccess.search_data(
    short_name=SHORT_NAME,
    bounding_box=BBOX,
    temporal=(START, END),
)
print(f"{len(results)} granules")
len(results)

In [ ]:
# Peek at the first granule's links + id.
if len(results):
    g = results[0]
    print("HTTPS:", g.data_links())
    print("S3   :", g.data_links(access="direct"))
    print("GranuleUR:", g["umm"].get("GranuleUR"))
else:
    print("No granules — check login, widen the date window, or verify the short_name.")

## 3. Build the submit list

The worker downloads the granule itself with MAAP/ASF temporary credentials (S3-preferred),
so we just pass the **`s3://` href** per granule — no pre-staging to a bucket. We collect
`(s3, https)` per result; the submit cell sends `s3_href` and lets the worker fall back to
authenticated HTTPS if needed.

In [ ]:
def first_h5(links):
    return next((u for u in (links or []) if u.lower().endswith(".h5")),
                (links[0] if links else None))

granules = []
for g in results:
    name = g["umm"].get("GranuleUR")
    granules.append({
        "name": name,
        "https": first_h5(g.data_links()),
        "s3": first_h5(g.data_links(access="direct")),
    })

print(f"{len(granules)} granules")
for gr in granules[:5]:
    print(" ", gr["name"])
    print("     s3   :", gr["s3"])

## 4. Resolve the deployed OGC process id

`maap.list_algorithms()` returns `{"processes": [{"processID": NN, "id": "nisar_access_subset", ...}]}`.
**`submit_job` needs the numeric `processID`** (e.g. `87`), *not* the string `id`
(`"nisar_access_subset"`) — passing the string (or a stale/deleted id) yields a 500 auth error.
We pick the newest matching `processID`.

In [ ]:
resp = maap.list_algorithms()
procs = resp.json().get("processes", []) if resp.status_code == 200 else []
for p in procs:
    print(p.get("processID"), "|", p.get("id"), "|", p.get("version"))

# submit_job wants the NUMERIC processID (not the string id). Grab the newest matching one.
matches = [p for p in procs
           if "nisar_access_subset" in str(p.get("id", "")).lower()
           or "nisar_access_subset" in str(p.get("title", "")).lower()]
PROCESS_ID = max((p["processID"] for p in matches), default=None)
print("\nPROCESS_ID =", PROCESS_ID)

## 5. Submit one DPS job per granule

Start with a **single** granule (`granules[:1]`) to validate the plumbing. We pass `s3_href`
(+ `https_href` fallback) and `access_mode="auto"`; the worker downloads the granule with
MAAP/ASF temporary credentials, reads `vars` from `group`, optionally subsets by
`bbox`/`bbox_crs`, and writes `out_name` (a Zarr store). `submit_job` returns HTTP 202 with
`{"id": ..., "status": "accepted"}`.

Defaults run on `maap-dps-worker-8gb` (the algorithm's `ram_min` is 8 GB). For a quick
plumbing check, switch to `maap-dps-sandbox` (8 GB / 10 min). The per-job subset `BBOX_SUBSET`
and its CRS are passed straight through to `nisar_access_subset.py`; leave `BBOX_SUBSET`
empty to keep the full granule extent.

In [ ]:
QUEUE = "maap-dps-worker-8gb"      # real-run queue (algorithm ram_min=8). Use
                                   # maap-dps-sandbox (8 GB, 10-min cap) for a quick plumbing
                                   # test, or maap-dps-worker-16gb for a heavy run.
TAG = "nisar-access-subset"

# Per-job subset passed to the worker (nisar_access_subset.py). These map directly to the
# algorithm inputs in algorithm.yml. Leave BBOX_SUBSET="" to keep the full granule.
VARS       = "HHHH"                                 # comma-separated freq-A variable names
GROUP      = "/science/LSAR/GCOV/grids/frequencyA"
BBOX_SUBSET = "750000,2450000,800000,2500000"      # minx,miny,maxx,maxy (in BBOX_CRS)
BBOX_CRS   = "EPSG:32643"                           # CRS/units of BBOX_SUBSET
OUT_NAME   = "nisar_subset.zarr"

SUBMIT = granules[:1]              # <-- widen to `granules` once the first job succeeds
assert PROCESS_ID, "PROCESS_ID not resolved — is the process deployed? (see §4)"
assert SUBMIT, "No granules — check the search cells."

rows = []
for i, gr in enumerate(SUBMIT, start=1):
    # Always submit the exact granule (its s3/https hrefs from the §2 AOI search).
    inputs = {
        "access_mode": "auto",     # S3 first, authenticated-HTTPS fallback
        "s3_href": gr["s3"],
        "https_href": gr["https"],
        "short_name": SHORT_NAME,
        "vars": VARS,
        "group": GROUP,
        "bbox": BBOX_SUBSET,
        "bbox_crs": BBOX_CRS,
        "out_name": OUT_NAME,
    }
    r = maap.submit_job(process_id=PROCESS_ID, inputs=inputs,   # PROCESS_ID is the int, e.g. 87
                        queue=QUEUE, dedup=True, tag=TAG)
    body = r.json() if r.status_code == 202 else {}
    job_id, status = body.get("jobID") or body.get("id"), body.get("status", r.text)
    print(f"[{i}/{len(SUBMIT)}] {r.status_code} job_id={job_id} status={status}")
    rows.append({"n": i, "granule": gr["name"], "s3_href": gr["s3"], "job_id": job_id,
                 "submit_status": status, "http": r.status_code,
                 "submit_time": datetime.datetime.now().isoformat()})

submit_df = pd.DataFrame(rows)
out_dir = os.path.expanduser("~/my-public-bucket/dps_submission_results")
os.makedirs(out_dir, exist_ok=True)
stamp = datetime.datetime.now().strftime("%Y%m%d%H%M")
csv_path = f"{out_dir}/nisar_access_subset_{TAG}_{stamp}.csv"
submit_df.to_csv(csv_path, index=False)
print("saved", csv_path)
submit_df

## 6. Monitor jobs and fetch results

`get_job_status` / `get_job_result` return JSON in maap-py v5. Results land under
`~/my-private-bucket/dps_output/nisar_access_subset/...` and contain the output Zarr store
plus `manifest.json`.

In [ ]:
for job_id in [j for j in submit_df["job_id"].tolist() if j]:
    s = maap.get_job_status(job_id)
    st = s.json().get("status") if s.status_code == 200 else s.text
    print(job_id, "->", st)

In [ ]:
# Once a job shows succeeded, inspect its outputs:
SUCCESS_JOB_ID = ""  # paste a job id
if SUCCESS_JOB_ID:
    r = maap.get_job_result(SUCCESS_JOB_ID)
    print(json.dumps(r.json(), indent=2) if r.status_code == 200 else r.text)
    m = maap.get_job_metrics(SUCCESS_JOB_ID)
    if m.status_code == 200:
        print(json.dumps(m.json(), indent=2))